<a href="https://colab.research.google.com/github/daffu081/Employee-Attrition-Analysis/blob/main/fill_station_names.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [4]:
# Step 1: Upload your files
from google.colab import files
uploaded = files.upload()

Saving Stations.csv to Stations (1).csv
Saving Toilets.csv to Toilets.csv


In [5]:
# Step 2: Load the CSVs
import pandas as pd

stations_df = pd.read_csv("Stations.csv")
toilets_df  = pd.read_csv("Toilets.csv")

print("Stations shape:", stations_df.shape)
print("Toilets shape: ", toilets_df.shape)
toilets_df[["Station", "StationUniqueId"]].head()

Stations shape: (509, 14)
Toilets shape:  (407, 46)


,Station,StationUniqueId
0,NaN,910GACTONML
1,NaN,910GBHILLPK
2,NaN,910GBHILLPK
3,NaN,910GBHILLPK
4,NaN,910GBNHAM


In [46]:
nan_names_count = stations_df['Name'].isna().sum()
print(f"Number of NaN values in 'Name' column of stations_df: {nan_names_count}")

if nan_names_count > 0:
    print(f"The {nan_names_count} NaN values in the 'Name' column account for the difference between the 509 rows in stations.csv and the 506 station sections in the JSON output. Stations without a valid name are not included in the JSON.")
else:
    print("There are no NaN values in the 'Name' column, so the discrepancy must be due to duplicate station names being counted as a single section in the JSON, or other filtering conditions.")

Number of NaN values in 'Name' column of stations_df: 0
There are no NaN values in the 'Name' column, so the discrepancy must be due to duplicate station names being counted as a single section in the JSON, or other filtering conditions.


In [48]:
unique_station_names_count = stations_df['Name'].nunique()
print(f"Number of unique station names in stations_df: {unique_station_names_count}")

if unique_station_names_count == len(output_json_data_cleaned):
    print("This confirms that the JSON output contains one section for each unique station name.")
else:
    print("There's still a mismatch even after considering unique names. Further investigation may be needed.")

Number of unique station names in stations_df: 506
This confirms that the JSON output contains one section for each unique station name.


In [64]:
def compare_df_to_json_section(df_name, dataframe, json_data_sample_entry, json_section_key, dropped_columns=None):
    print(f"--- Checking {df_name} against '{json_section_key}' section ---")
    if dropped_columns is None:
        dropped_columns = []

    # Get expected columns from the original DataFrame (excluding dropped ones)
    expected_cols_df = set(dataframe.columns) - set(dropped_columns)

    actual_keys_json = set()

    if json_section_key == 'Stations':
        # 'Stations' is now a list of dictionary entries
        if isinstance(json_data_sample_entry, list) and len(json_data_sample_entry) > 0:
            # Take keys from the first item in the list for comparison
            actual_keys_json = set(json_data_sample_entry[0].keys())
    else:
        # Other sections are lists of dictionaries, we check the first item if the list is not empty
        if isinstance(json_data_sample_entry, list) and len(json_data_sample_entry) > 0:
            actual_keys_json = set(json_data_sample_entry[0].keys())
        # If it's not a list, it might be an empty dictionary if no data for the section
        elif isinstance(json_data_sample_entry, dict) and json_data_sample_entry:
            actual_keys_json = set(json_data_sample_entry.keys())

    # Handle cases where the JSON section might be empty for the sample station
    if not actual_keys_json and not expected_cols_df:
        print(f"  Both DataFrame {df_name} (after dropping) and JSON section '{json_section_key}' are empty for sample. No columns to compare.")
        print("-" * 50)
        return
    elif not actual_keys_json and expected_cols_df:
        print(f"  Warning: DataFrame {df_name} has expected columns: {expected_cols_df}, but JSON section '{json_section_key}' is empty for sample station. This might indicate no matching data for the sample, or columns were not included.")
        print("-" * 50)
        return
    elif actual_keys_json and not expected_cols_df:
        print(f"  Warning: JSON section '{json_section_key}' has keys: {actual_keys_json}, but DataFrame {df_name} (after dropping) has no expected columns.")
        print("-" * 50)
        return

    # Compare
    missing_in_json = expected_cols_df - actual_keys_json
    extra_in_json = actual_keys_json - expected_cols_df

    if not missing_in_json and not extra_in_json:
        print(f"  All expected columns from {df_name} are present in '{json_section_key}' section of JSON (considering dropped columns).")
    else:
        if missing_in_json:
            print(f"  Columns from {df_name} missing in '{json_section_key}' section of JSON: {missing_in_json}")
        if extra_in_json:
            print(f"  Columns in '{json_section_key}' section of JSON not found in {df_name} (after dropping): {extra_in_json}")

    print("-" * 50)

# Choose a sample station that is likely to have data in all sections for a comprehensive check
# With the new 509 entry structure, the top-level keys are UniqueIds. We need to pick one.
sample_station_name = 'Abbey Wood' # Example UniqueId for Abbey Wood

if sample_station_name not in output_json_data_cleaned:
    print(f"Error: Sample station UniqueId '{sample_station_name}' not found in the cleaned JSON output. Cannot perform column checks.")
else:
    sample_json_station_data = output_json_data_cleaned[sample_station_name]

    # 1. Check Stations section
    # All columns from stations_df are intended to be in the JSON 'Stations' section
    compare_df_to_json_section(
        'stations_df', stations_df,
        sample_json_station_data.get('Stations', []), # 'StationDetails' is a dict
        'Stations',
        # 'UniqueId' and 'Name' are now part of StationDetails, so no longer dropped if they are there.
        # 'Name' is used for filtering, but also present in StationDetails.
        # Only drop columns that truly aren't expected in the final StationDetails sub-object
        # Based on the JSON generation logic, all stations_df columns (except for the ones set as top-level key) are present.
        # No columns to explicitly drop from `stations_df` when comparing to `StationDetails` as all are mapped.
        dropped_columns=[]
    )

    # The remaining sections (Lifts, Platforms, etc.) are still aggregated by station name.
    # The previous logic for these should still apply, as they are sub-lists of dictionaries.

    # 2. Check Lifts section
    # 'Station' and 'StationUniqueId' are explicitly dropped from lifts_df before conversion
    compare_df_to_json_section(
        'lifts_df', lifts_df,
        sample_json_station_data.get('Lifts', []), # Pass a list for 'Lifts'
        'Lifts',
        dropped_columns=['Station', 'StationUniqueId']
    )

    # 3. Check Platforms section
    # 'Station' and 'StationUniqueId' are explicitly dropped from platforms_df before conversion
    # Note: FriendlyName was added to platforms_df and should be present if not explicitly dropped
    compare_df_to_json_section(
        'platforms_df', platforms_df,
        sample_json_station_data.get('Platforms', []), # Pass a list for 'Platforms'
        'Platforms',
        dropped_columns=['Station', 'StationUniqueId']
    )

    # 4. Check Toilets section
    # 'Station' and 'StationUniqueId' are explicitly dropped from toilets_df before conversion
    compare_df_to_json_section(
        'toilets_df', toilets_df,
        sample_json_station_data.get('Toilet', []), # Pass a list for 'Toilet'
        'Toilet',
        dropped_columns=['Station', 'StationUniqueId']
    )

    # 5. Check StationPoints section
    # 'Station' and 'StationUniqueId' are explicitly dropped from stationpoints_df before conversion
    compare_df_to_json_section(
        'stationpoints_df', stationpoints_df,
        sample_json_station_data.get('StationPoints', []), # Pass a list for 'StationPoints'
        'StationPoints',
        dropped_columns=['Station', 'StationUniqueId']
    )

    # 6. Check PlatformServices section
    # 'Station' and 'StationUniqueId' are explicitly dropped from platformservices_df before conversion
    compare_df_to_json_section(
        'platformservices_df', platformservices_df,
        sample_json_station_data.get('PlatformServices', []), # Pass a list for 'PlatformServices'
        'PlatformServices',
        dropped_columns=['Station', 'StationUniqueId']
    )

--- Checking stations_df against 'Stations' section ---
  All expected columns from stations_df are present in 'Stations' section of JSON (considering dropped columns).
--------------------------------------------------
--- Checking lifts_df against 'Lifts' section ---
  All expected columns from lifts_df are present in 'Lifts' section of JSON (considering dropped columns).
--------------------------------------------------
--- Checking platforms_df against 'Platforms' section ---
  All expected columns from platforms_df are present in 'Platforms' section of JSON (considering dropped columns).
--------------------------------------------------
--- Checking toilets_df against 'Toilet' section ---
  All expected columns from toilets_df are present in 'Toilet' section of JSON (considering dropped columns).
--------------------------------------------------
--- Checking stationpoints_df against 'StationPoints' section ---
  All expected columns from stationpoints_df are present in 'Station

In [47]:
# Step 3: Fill the Station column
id_to_name = stations_df.set_index("UniqueId")["Name"].to_dict()

missing_before = toilets_df["Station"].isna().sum()
toilets_df["Station"] = toilets_df["StationUniqueId"].map(id_to_name)
missing_after = toilets_df["Station"].isna().sum()

print(f"Total rows        : {len(toilets_df)}")
print(f"Station names filled : {missing_before - missing_after}")
print(f"Rows still missing   : {missing_after}")

unmatched = toilets_df[toilets_df["Station"].isna()]["StationUniqueId"].unique()
if len(unmatched) > 0:
    print(f"Unmatched IDs: {list(unmatched)}")

toilets_df[["Station", "StationUniqueId"]].head(10)

Total rows        : 407
Station names filled : 0
Rows still missing   : 0


,Station,StationUniqueId
0,Acton Main Line,910GACTONML
1,Bush Hill Park,910GBHILLPK
2,Bush Hill Park,910GBHILLPK
3,Bush Hill Park,910GBHILLPK
4,Burnham,910GBNHAM
5,Burnham,910GBNHAM
6,Burnham,910GBNHAM
7,Brentwood,910GBRTWOOD
8,Brentwood,910GBRTWOOD
9,Brentwood,910GBRTWOOD


In [7]:
# Step 4: Save and download the updated CSV
output_path = "Toilets_updated.csv"
toilets_df.to_csv(output_path, index=False)
print(f"Saved: {output_path}")

files.download(output_path)

Saved: Toilets_updated.csv


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [8]:
uploaded = files.upload()

Saving Lifts.csv to Lifts.csv


In [9]:
lifts_df  = pd.read_csv("Lifts.csv")

In [11]:
# Step 3: Fill the Station column
id_to_name = stations_df.set_index("UniqueId")["Name"].to_dict()

missing_before = lifts_df["Station"].isna().sum()
lifts_df["Station"] = lifts_df["StationUniqueId"].map(id_to_name)
missing_after = lifts_df["Station"].isna().sum()

print(f"Total rows        : {len(lifts_df)}")
print(f"Station names filled : {missing_before - missing_after}")
print(f"Rows still missing   : {missing_after}")

unmatched = lifts_df[lifts_df["Station"].isna()]["StationUniqueId"].unique()
if len(unmatched) > 0:
    print(f"Unmatched IDs: {list(unmatched)}")

lifts_df[["Station", "StationUniqueId"]].head(10)

Total rows        : 569
Station names filled : 569
Rows still missing   : 0


,Station,StationUniqueId
0,Acton Main Line,910GACTONML
1,Acton Main Line,910GACTONML
2,Brent Cross West Station,910GBRENTX
3,Brent Cross West Station,910GBRENTX
4,Brent Cross West Station,910GBRENTX
5,Brent Cross West Station,910GBRENTX
6,Brent Cross West Station,910GBRENTX
7,Brockley,910GBROCKLY
8,Bromley South,910GBROMLYS
9,Bromley South,910GBROMLYS


In [27]:
# Step 4: Save and download the updated CSV
output_path = "Lifts_updated.csv"
lifts_df.to_csv(output_path, index=False)
print(f"Saved: {output_path}")

files.download(output_path)

Saved: Lifts_updated.csv


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [16]:
uploaded = files.upload()

Saving Platforms.csv to Platforms (1).csv


In [17]:
platforms_df  = pd.read_csv("Platforms.csv")

In [19]:
# Step 3: Fill the Station column
id_to_name = stations_df.set_index("UniqueId")["Name"].to_dict()

# Initialize missing_before to the total number of rows since 'Station' column is being created
missing_before = len(platforms_df)

platforms_df["Station"] = platforms_df["StationUniqueId"].map(id_to_name)
missing_after = platforms_df["Station"].isna().sum()

print(f"Total rows        : {len(platforms_df)}")
print(f"Station names filled : {missing_before - missing_after}")
print(f"Rows still missing   : {missing_after}")

unmatched = platforms_df[platforms_df["Station"].isna()]["StationUniqueId"].unique()
if len(unmatched) > 0:
    print(f"Unmatched IDs: {list(unmatched)}")

platforms_df[["Station", "StationUniqueId"]].head(10)

Total rows        : 1584
Station names filled : 1584
Rows still missing   : 0


,Station,StationUniqueId
0,Abbey Wood,HUBABW
1,Abbey Wood,HUBABW
2,Abbey Wood,HUBABW
3,Abbey Wood,HUBABW
4,Acton Central,910GACTNCTL
5,Acton Central,910GACTNCTL
6,Acton Main Line,910GACTONML
7,Acton Main Line,910GACTONML
8,Anerley,910GANERLEY
9,Anerley,910GANERLEY


In [21]:
# Step 4: Save and download the updated CSV
output_path = "Platforms_updated.csv"
platforms_df.to_csv(output_path, index=False)
print(f"Saved: {output_path}")

files.download(output_path)

Saved: Platforms_updated.csv


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [28]:
uploaded = files.upload()

Saving StationPoints.csv to StationPoints (1).csv


In [29]:
stationpoints_df = pd.read_csv("StationPoints.csv")

In [31]:
# Step 3: Fill the Station column
id_to_name = stations_df.set_index("UniqueId")["Name"].to_dict()

# Initialize missing_before to the total number of rows since 'Station' column is being created
missing_before = len(stationpoints_df)

stationpoints_df["Station"] = stationpoints_df["StationUniqueId"].map(id_to_name)
missing_after = stationpoints_df["Station"].isna().sum()

print(f"Total rows        : {len(stationpoints_df)}")
print(f"Station names filled : {missing_before - missing_after}")
print(f"Rows still missing   : {missing_after}")

unmatched = stationpoints_df[stationpoints_df["Station"].isna()]["StationUniqueId"].unique()
if len(unmatched) > 0:
    print(f"Unmatched IDs: {list(unmatched)}")

stationpoints_df[["Station", "StationUniqueId"]].head(10)

Total rows        : 4084
Station names filled : 4084
Rows still missing   : 0


,Station,StationUniqueId
0,Acton Central,910GACTNCTL
1,Acton Central,910GACTNCTL
2,Acton Central,910GACTNCTL
3,Acton Central,910GACTNCTL
4,Acton Central,910GACTNCTL
5,Acton Central,910GACTNCTL
6,Acton Central,910GACTNCTL
7,Acton Main Line,910GACTONML
8,Acton Main Line,910GACTONML
9,Acton Main Line,910GACTONML


In [32]:
# Step 4: Save and download the updated CSV
output_path = "StationPoints_updated.csv"
stationpoints_df.to_csv(output_path, index=False)
print(f"Saved: {output_path}")

files.download(output_path)

Saved: StationPoints_updated.csv


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [33]:
uploaded = files.upload()

Saving PlatformServices.csv to PlatformServices.csv


In [36]:
# Step 2: Load PlatformServices.csv
platformservices_df = pd.read_csv("PlatformServices.csv")

print("PlatformServices shape (before merge):", platformservices_df.shape)

# Create 'PlatformName' in platforms_df by copying 'UniqueId'
# This is done because 'PlatformName' was requested but not present in platforms_df columns.
platforms_df['FriendlyName'] = platforms_df['UniqueId']

# Step 3: Merge with platforms_df to add required columns
# Assuming PlatformUniqueId in PlatformServices.csv matches UniqueId in platforms_df
platformservices_df = pd.merge(
    platformservices_df,
    platforms_df[['UniqueId', 'FriendlyName', 'PlatformNumber', 'Station', 'StationUniqueId']],
    left_on='PlatformUniqueId',
    right_on='UniqueId',
    how='left'
)

# Drop the redundant 'UniqueId' column from the merge if it exists
platformservices_df = platformservices_df.drop(columns=['UniqueId'], errors='ignore')

print("PlatformServices shape (after merge):", platformservices_df.shape)
print("PlatformServices with new columns (head):")
print(platformservices_df[['PlatformUniqueId', 'FriendlyName', 'PlatformNumber', 'Station', 'StationUniqueId']].head())

# Step 4: Save and download the updated CSV
output_path = "PlatformServices_updated.csv"
platformservices_df.to_csv(output_path, index=False)
print(f"Saved: {output_path}")

files.download(output_path)

PlatformServices shape (before merge): (1878, 15)
PlatformServices shape (after merge): (1878, 19)
PlatformServices with new columns (head):
                          PlatformUniqueId  \
0           HUBABW-Plat01-WB-national-rail   
1           HUBABW-Plat02-EB-national-rail   
2               HUBABW-Plat03-WB-elizabeth   
3               HUBABW-Plat04-WB-elizabeth   
4  910GACTNCTL-Plat01-WB-london-overground   

                              FriendlyName PlatformNumber        Station  \
0           HUBABW-Plat01-WB-national-rail              1     Abbey Wood   
1           HUBABW-Plat02-EB-national-rail              2     Abbey Wood   
2               HUBABW-Plat03-WB-elizabeth              3     Abbey Wood   
3               HUBABW-Plat04-WB-elizabeth              4     Abbey Wood   
4  910GACTNCTL-Plat01-WB-london-overground              1  Acton Central   

  StationUniqueId  
0          HUBABW  
1          HUBABW  
2          HUBABW  
3          HUBABW  
4     910GACTNCTL  
Saved

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [63]:
import json
import pandas as pd
import numpy as np
from google.colab import files

def replace_nan_with_none(obj):
    if isinstance(obj, dict):
        return {k: replace_nan_with_none(v) for k, v in obj.items()}
    elif isinstance(obj, list):
        return [replace_nan_with_none(elem) for elem in obj]
    # Handle numpy booleans
    elif isinstance(obj, (np.bool_, bool)):
        return bool(obj)
    elif (isinstance(obj, float) and np.isnan(obj)) or (isinstance(obj, str) and obj == 'NaN'):
        return None
    return obj

output_json_data = {}

# Iterate through each row of stations_df to get 509 entries
for idx, row in stations_df.iterrows():
    station_unique_id = row['UniqueId']
    station_name = row['Name'] # Get the station name for filtering other dataframes

    # Initialize the entry for the station name if it doesn't exist as a list
    if station_name not in output_json_data:
        output_json_data[station_name] = {
            'Stations': [], # This will now be a list of station details
            'Lifts': [],
            'Platforms': [],
            'Toilet': [],
            'StationPoints': [],
            'PlatformServices': []
        }

    # Append current station's details to the 'Stations' list
    output_json_data[station_name]['Stations'].append({
        'Name': station_name,
        'UniqueId': station_unique_id,
        'FareZones': str(row['FareZones']),
        'HubNaptanCode': row['HubNaptanCode'],
        'Wifi': row['Wifi'],
        'OutsideStationUniqueId': row['OutsideStationUniqueId'],
        'BlueBadgeCarParking': row['BlueBadgeCarParking'],
        'BlueBadgeCarParkSpaces': row['BlueBadgeCarParkSpaces'],
        'TaxiRanksOutsideStation': row['TaxiRanksOutsideStation'],
        'MainBusInterchange': row['MainBusInterchange'],
        'PierInterchange': row['PierInterchange'],
        'NationalRailInterchange': row['NationalRailInterchange'],
        'AirportInterchange': row['AirportInterchange'],
        'EmiratesAirLineInterchange': row['EmiratesAirLineInterchange'],
    })

    # Extract Lifts information (still aggregated by station_name)
    lifts_for_station = lifts_df[lifts_df['Station'] == station_name]
    # Extend the list with new data for the current station
    output_json_data[station_name]['Lifts'].extend(
        lifts_for_station.drop(columns=['Station', 'StationUniqueId'], errors='ignore').to_dict(orient='records')
    )

    # Extract Platforms information (still aggregated by station_name)
    platforms_for_station = platforms_df[platforms_df['Station'] == station_name]
    output_json_data[station_name]['Platforms'].extend(
        platforms_for_station.drop(columns=['Station', 'StationUniqueId'], errors='ignore').to_dict(orient='records')
    )

    # Extract Toilets information (still aggregated by station_name)
    toilets_for_station = toilets_df[toilets_df['Station'] == station_name]
    output_json_data[station_name]['Toilet'].extend(
        toilets_for_station.drop(columns=['Station', 'StationUniqueId'], errors='ignore').to_dict(orient='records')
    )

    # Extract StationPoints information (still aggregated by station_name)
    stationpoints_for_station = stationpoints_df[stationpoints_df['Station'] == station_name]
    output_json_data[station_name]['StationPoints'].extend(
        stationpoints_for_station.drop(columns=['Station', 'StationUniqueId'], errors='ignore').to_dict(orient='records')
    )

    # Extract PlatformServices information (still aggregated by station_name)
    platformservices_for_station = platformservices_df[platformservices_df['Station'] == station_name]
    output_json_data[station_name]['PlatformServices'].extend(
        platformservices_for_station.drop(columns=['Station', 'StationUniqueId'], errors='ignore').to_dict(orient='records')
    )

# Apply the NaN to null replacement function
output_json_data_cleaned = replace_nan_with_none(output_json_data)

# Save the dictionary to a JSON file
json_output_filename = "station_details.json"
with open(json_output_filename, 'w') as f:
    json.dump(output_json_data_cleaned, f, indent=4)

print(f"Generated {json_output_filename}")

# Download the JSON file
files.download(json_output_filename)

Generated station_details.json


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [65]:
duplicate_station_names = stations_df[stations_df.duplicated(subset=['Name'], keep=False)]
print("Station names appearing more than once in Stations.csv:")
for name in duplicate_station_names['Name'].unique():
    print(f"- {name}")

# Display the full rows for these duplicate stations
print("\nDetails for duplicate station names:")
display(duplicate_station_names.sort_values(by='Name'))

Station names appearing more than once in Stations.csv:
- Bethnal Green
- West Hampstead
- Edgware Road

Details for duplicate station names:


,UniqueId,Name,FareZones,HubNaptanCode,Wifi,OutsideStationUniqueId,BlueBadgeCarParking,BlueBadgeCarParkSpaces,TaxiRanksOutsideStation,MainBusInterchange,PierInterchange,NationalRailInterchange,AirportInterchange,EmiratesAirLineInterchange
16,910GBTHNLGR,Bethnal Green,2,NaN,True,910GBTHNLGR-Outside,False,NaN,False,NaN,NaN,NaN,NaN,NaN
228,940GZZLUBLG,Bethnal Green,2,NaN,True,940GZZLUBLG-Outside,False,NaN,False,NaN,NaN,NaN,NaN,NaN
271,940GZZLUERB,Edgware Road,1,NaN,True,940GZZLUERB-Outside,False,NaN,False,NaN,NaN,NaN,NaN,NaN
272,940GZZLUERC,Edgware Road,1,NaN,True,940GZZLUERC-Outside,False,NaN,False,NaN,NaN,NaN,NaN,NaN
141,910GWHMDSTD,West Hampstead,2,HUBWHD,True,910GWHMDSTD-Outside,False,NaN,False,NaN,NaN,NaN,NaN,NaN
401,940GZZLUWHP,West Hampstead,2,HUBWHD,True,940GZZLUWHP-Outside,False,NaN,False,NaN,NaN,NaN,NaN,NaN


In [66]:
print(f"Number of station entries in station_details.json: {len(output_json_data_cleaned)}")

Number of station entries in station_details.json: 506


In [68]:
print(f"Number of station entries in station_details.json: {len(output_json_data_cleaned)}")

Number of station entries in station_details.json: 506


### `GetTramStationAccessibility` Function

This `async` function allows you to retrieve accessibility data for a specific station. You provide the `station_name` and a list of `tags` (e.g., 'Lifts', 'Toilets', 'Platforms', 'StationPoints', 'PlatformServices', 'Stations') to specify which types of information you want. The function will return a dictionary with a `status` indicating if the station was found and the requested `data`.

In [70]:
import asyncio

async def GetTramStationAccessibility(station_name: str, tags: list):

    if station_name not in output_json_data_cleaned:
        return {"status": "not found", "data": {}}

    station_data = output_json_data_cleaned[station_name]
    result_data = {}

    for tag in tags:
        # Ensure the tag corresponds to a key in station_data and return an empty list if not found
        result_data[tag] = station_data.get(tag, [])

    return {"status": "found", "data": result_data}

